# Microsoft Agent Framework — Azure OpenAI（Responses API）

在此代码示例中，您将使用 **Microsoft Agent Framework (MAF)** 来创建一个由 **Azure OpenAI** 支持的简单代理，使用的是 **Responses API**。

> **迁移说明：** 该示例之前使用了 Semantic Kernel 和 GitHub Models。现在已迁移到 Microsoft Agent Framework，GitHub Models（已弃用，将于 2026 年 7 月退役）被 Azure OpenAI 所取代，Azure OpenAI 支持 Responses API。MAF 中的 `OpenAIChatClient` 目标是 Azure OpenAI 的稳定 `/openai/v1/` 端点，默认使用 Responses API。

本示例的目的是演示后续其他代码示例中实现各种智能代理模式时将应用的步骤。


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## 导入所需的 Python 包


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## 定义工具

在 Microsoft Agent Framework 中，<strong>工具</strong> 是一个使用 `@tool` 装饰的普通 Python 函数，代理可以调用它。下面我们定义了一个工具，该工具返回一个随机的度假目的地，并避免重复上一次的选择。


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## 创建代理

在这里，我们创建名为 `TravelAgent` 的代理。

在此示例中，我们使用非常基本的指令。您可以自由修改这些指令，以观察代理行为的变化。


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## 运行代理

现在我们可以运行代理了。我们创建一个 `AgentSession`，这样代理就能记住跨轮的对话，然后发送两个 `user_inputs`。第一个请求一个旅行计划；第二个表示用户不喜欢建议，并请求另一个——代理使用会话历史加上 `get_random_destination` 工具进行回应。

你可以修改这些消息，观察代理的不同反应。响应是按<strong>令牌逐个</strong>流式传输的。


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
